# 01 - Deploy the Foundry infrastructure

Run **00_setup** first and select the demo kernel. This notebook deploys one Foundry account/project and all three selected models, then generates non-secret Entra connection settings. GPT-5.4 quota has been freed and the user approved restoring the intermediate target. Platform behavior belongs in Bicep, not notebook workarounds.

Milestone 2 is implemented. Existing approved tests/results are retained when adding a target; do not regenerate or reevaluate completed targets automatically. Global Standard may process requests outside the account region. Review [costs, constraints, and cleanup](../README.md) first.

## Load the demo tools

Open this notebook from its `notebooks` directory, as with the standard setup/destroy notebooks. No connection secrets belong in these cells.

In [ ]:
import json
import os
import subprocess
import sys
from datetime import datetime, timezone
from pathlib import Path

from azure.identity import AzureCliCredential
from azure.mgmt.cognitiveservices import CognitiveServicesManagementClient
from azure.mgmt.resource.resources import ResourceManagementClient
import pandas as pd

demo_root = Path('..').resolve()
sys.path.insert(0, str(demo_root))
from app.assert_lab import DeploymentMetadata, ModelDeployment, artifact_path, validate_catalog, write_connections
from app.notebook_helpers import resolve_az_cli

az_cmd = resolve_az_cli()
if az_cmd is None:
    raise FileNotFoundError('Azure CLI is required; run the setup notebook first.')
generated = artifact_path(root=demo_root)
generated.mkdir(parents=True, exist_ok=True)
print(f'Generated artifacts: {generated}')

## Confirm the subscription and three model selections

Set `AZURE_SUBSCRIPTION_ID` or edit `SUBSCRIPTION_ID` explicitly using the subscription list from setup. No first/default subscription is selected automatically. For a new group, confirm its subscription before continuing.

The following dated model selections were researched and checked against the live catalog during authoring; they are editable requests, not guarantees of future availability. Each capacity is 10 model-specific allocation units, the catalog's advertised default at authoring, **not an assumed universal TPM amount**. The next step checks exact versions, lifecycle, SKU, capacity, and current quota before creating any resources. Do not silently substitute a retired, unavailable, or blocked model.

In [ ]:
SUBSCRIPTION_ID = os.environ.get('AZURE_SUBSCRIPTION_ID', '')
RESOURCE_GROUP = 'rg-assert01'
LOCATION = 'eastus2'
PROVIDER_API_VERSION = '2024-10-21'
MODEL_PLAN = (
    ModelDeployment('older', 'assert-older', 'gpt-5', '2025-08-07', 'GlobalStandard', 10),
    ModelDeployment('intermediate', 'assert-intermediate', 'gpt-5.4', '2026-03-05', 'GlobalStandard', 10),
    ModelDeployment('newer', 'assert-newer', 'gpt-5.6-sol', '2026-07-09', 'GlobalStandard', 10),
)
SKIPPED_MODEL = None
SKIP_REASON = None
MODEL_SELECTIONS = tuple(model for model in MODEL_PLAN if model != SKIPPED_MODEL)
if not SUBSCRIPTION_ID.strip():
    raise ValueError('Set SUBSCRIPTION_ID explicitly or set AZURE_SUBSCRIPTION_ID before provisioning.')
credential = AzureCliCredential()
cognitive = CognitiveServicesManagementClient(credential, SUBSCRIPTION_ID)
resources = ResourceManagementClient(credential, SUBSCRIPTION_ID)
print(f'Selected subscription: {SUBSCRIPTION_ID}; group: {RESOURCE_GROUP}; region: {LOCATION}')
if SKIPPED_MODEL is not None:
    print(f'Explicitly skipped {SKIPPED_MODEL.alias}: {SKIP_REASON}')

## Check the live model catalog and quota

This is a required deployment prerequisite, not a policy evaluation. Insufficient quota stops here before a group/account is created. Free unused allocation or request more quota, then rerun; selecting another model/subscription/region requires an explicit decision. No automatic fallback or quota change is performed.

In [ ]:
catalog = list(cognitive.models.list(LOCATION))
usages = list(cognitive.usages.list(LOCATION))
model_checks = validate_catalog(MODEL_SELECTIONS, catalog, usages, skipped=SKIPPED_MODEL)
display(pd.DataFrame(model_checks))

## Review inherited policy before provisioning

Follow the repository's [scoped Azure Policy review procedure](../../../.github/skills/azure-demo-authoring/references/workflow.md#scoped-azure-policy-review) for the confirmed subscription and intended resource-group scope. Include inherited assignments/exemptions, initiatives/member definitions, effective parameters/effects, exclusions, overrides, and enforcement.

Confirm policy permits the Bicep-defined **public endpoint with local authentication disabled**, selected SKU/region, and group/resource creation. The approved Entra path complies with inherited policy that disables local authentication; no exclusion tags or exemptions are added. Record non-secret review evidence in the architecture notes, then acknowledge below. Repeat the scope review for the actual group before deploying resources.

In [ ]:
POLICY_REVIEW_COMPLETE = False
if not POLICY_REVIEW_COMPLETE:
    raise ValueError('Complete the scoped inherited-policy review above before provisioning; then acknowledge it here.')
preflight_path = artifact_path('preflight.json', root=demo_root)
preflight_path.write_text(json.dumps({
    'checked_at': datetime.now(timezone.utc).isoformat(),
    'subscription_id': SUBSCRIPTION_ID, 'resource_group': RESOURCE_GROUP,
    'location': LOCATION, 'policy_review_complete': True, 'models': model_checks,
}, indent=2) + '\n', encoding='utf-8')
print(f'Non-secret prerequisite record: {preflight_path}')

## Prepare the dedicated resource group

Only continue after the model/quota and policy prerequisites pass. This uses the management SDK; platform settings remain in Bicep.

In [ ]:
resource_group = resources.resource_groups.create_or_update(RESOURCE_GROUP, {'location': LOCATION})
print(f'Resource group ready: {resource_group.name}')

## Identify the signed-in developer for inference access

Select the same subscription in **Step 2** of setup first. Bicep grants this developer the account-scoped **Cognitive Services OpenAI User** role. You need permission to create role assignments. Use the existing CLI's signed-in-user lookup rather than adding a Graph SDK; no keys or bearer tokens are retrieved or stored.

In [ ]:
context = subprocess.run([az_cmd, 'account', 'show', '--query', 'id', '--output', 'tsv'], capture_output=True, text=True)
if context.returncode != 0:
    raise RuntimeError(f'Azure context lookup failed\nstderr: {context.stderr.strip()}\nstdout: {context.stdout.strip()}')
if context.stdout.strip() != SUBSCRIPTION_ID:
    raise ValueError('Select the confirmed subscription in setup before deploying or using native Entra authentication.')
principal = subprocess.run([az_cmd, 'ad', 'signed-in-user', 'show', '--query', 'id', '--output', 'tsv'], capture_output=True, text=True)
if principal.returncode != 0:
    raise RuntimeError(f'Signed-in-user lookup failed\nstderr: {principal.stderr.strip()}\nstdout: {principal.stdout.strip()}')
INFERENCE_PRINCIPAL_ID = principal.stdout.strip()
if not INFERENCE_PRINCIPAL_ID:
    raise ValueError('The signed-in developer lookup returned no object ID.')
print('The signed-in developer will receive account-scoped inference access through Bicep.')

## Build and deploy the copied Bicep

Versions are explicit inputs; Bicep has no model-version defaults. It creates one account/project, the explicitly selected two or three deployments, and developer inference RBAC. Platform settings are Entra-only authentication (`disableLocalAuth: true`), public access, Global Standard, and `NoAutoUpgrade`. Only non-secret inputs/outputs pass through CLI.

In [ ]:
template_path = demo_root / 'infra' / 'main.bicep'
parameters_path = artifact_path('deployment.parameters.json', root=demo_root)
parameters_path.write_text(json.dumps({
    '$schema': 'https://schema.management.azure.com/schemas/2019-04-01/deploymentParameters.json#',
    'contentVersion': '1.0.0.0',
    'parameters': {
        'location': {'value': LOCATION},
        'inferencePrincipalId': {'value': INFERENCE_PRINCIPAL_ID},
        'modelDeployments': {'value': [model.to_dict() for model in MODEL_SELECTIONS]},
    },
}, indent=2) + '\n', encoding='utf-8')
build_result = subprocess.run(
    [az_cmd, 'bicep', 'build', '--file', str(template_path), '--outfile', str(artifact_path('main.json', root=demo_root))],
    capture_output=True, text=True, cwd=demo_root,
)
if build_result.returncode != 0:
    raise RuntimeError(f'az bicep build failed (exit {build_result.returncode})\nstderr: {build_result.stderr.strip()}\nstdout: {build_result.stdout.strip()}')
if build_result.stderr.strip():
    print(build_result.stderr.strip())
print('Bicep build succeeded; this does not prove deployment or provider compatibility.')

In [ ]:
deployment_result = subprocess.run(
    [az_cmd, 'deployment', 'group', 'create', '--subscription', SUBSCRIPTION_ID,
     '--resource-group', RESOURCE_GROUP, '--name', 'assert01-infrastructure',
     '--template-file', str(template_path), '--parameters', '@' + str(parameters_path), '--output', 'json'],
    capture_output=True, text=True, cwd=demo_root,
)
if deployment_result.returncode != 0:
    raise RuntimeError(f'az deployment group create failed (exit {deployment_result.returncode})\nstderr: {deployment_result.stderr.strip()}\nstdout: {deployment_result.stdout.strip()}')
deployment_outputs = json.loads(deployment_result.stdout)['properties']['outputs']
print('Infrastructure deployment succeeded. Export actual metadata next; no account keys were ARM outputs.')

## Generate non-secret connection settings

Read actual deployed identities, versions, and capacities through the management SDK. No account keys are fetched. ASSERT uses the signed-in Azure CLI identity and its native refreshable Entra token provider. Both generated connection files contain only non-secret settings; do not display raw provider objects or tokens.

In [ ]:
account_name = deployment_outputs['aiServicesAccountName']['value']
account = cognitive.accounts.get(RESOURCE_GROUP, account_name)
if account.properties.disable_local_auth is not True or account.properties.public_network_access != 'Enabled':
    raise ValueError('Deployed authentication/network settings differ from Bicep; resolve upstream policy/template drift before connecting.')
actual_models = []
for selected in MODEL_SELECTIONS:
    actual = cognitive.deployments.get(RESOURCE_GROUP, account_name, selected.deployment_name)
    observed = ModelDeployment(selected.alias, actual.name, actual.properties.model.name, actual.properties.model.version, actual.sku.name, actual.sku.capacity)
    if observed != selected or actual.properties.version_upgrade_option != 'NoAutoUpgrade':
        raise ValueError(f'Actual deployment differs from the explicit selection: {selected.alias}')
    actual_models.append(observed)
metadata = DeploymentMetadata(
    subscription_id=SUBSCRIPTION_ID, resource_group=RESOURCE_GROUP, location=account.location,
    account_name=account.name, account_id=account.id, ai_endpoint=account.properties.endpoint,
    project_name=deployment_outputs['foundryProjectName']['value'],
    project_endpoint=deployment_outputs['foundryProjectEndpoint']['value'],
    api_base=deployment_outputs['azureOpenAIEndpoint']['value'], api_version=PROVIDER_API_VERSION,
    deployments=tuple(actual_models), skipped_deployment=SKIPPED_MODEL, skip_reason=SKIP_REASON,
)
env_path, metadata_path = write_connections(metadata, root=demo_root)
print(f'Generated ignored non-secret environment: {env_path}')
print(f'Generated non-secret metadata: {metadata_path}')

## Continue with Connect models

Open [02_configure.ipynb](./02_configure.ipynb) and complete the native text/structured-output smoke checks for all three models. Before M1 handoff, inspect execution outputs (including this notebook) for credentials, then run the existing notebook-cleanup hooks. Keep the environment for M2 and **stop after Milestone 1**. Deployment/build success alone does not pass M1.